# Introduction
GOAL: To train a model to predict the wild fire risk of properties, using census data as input features and the proximity to fires as a target feature.

It would be cool to add in data regarding local climate.

In [1]:
import pandas as pd
import load_wildfires
import load_census
import load_properties
import gis
import train
import visualize
import sqlalchemy as s

from pathlib import Path
from sql_funcs import SQL

from settings import PATH_DATA

In [2]:
SQL.kill_idle(True)
sql_obj = SQL()


kill_idle: terminated 8 connection(s)


# Load Properties

## Select Properties of Interest

Chosing 300000 properties randomly from US addresses. We will join relevant census data to these addresses. This will probably take awhile, so best to run it overnight.

We don't care about the address itself. We add a census identifier called the GEOID which based on the coordinate's state, county, and tract number.

Using a package that makes use of the [US Census Geocoder API](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/census-geocoder.html), requests can be in batches of 10,000.

https://pypi.org/project/random-address/

In [3]:
props = load_properties.Properties(sql_obj=sql_obj)


166420 properties loaded.


In [4]:
# result = sql_obj.connection.execute(s.text("SELECT property_id, geoid FROM properties LIMIT 5;"))
# for row in result:
#   print(row)
# raise

In [5]:
properties = props.get_properties_gpd()

properties.head()

,property_id,geoid,block_id,block_grp,tract_id,county_id,state_id,geometry
0,220250,360259705012013,2013,2,970501,25,36,POINT (1679681.993 2303127.485)
1,220251,270753701012282,2282,2,370101,75,27,POINT (337884.435 2694375.859)
2,220252,040159501032038,2038,2,950103,15,4,POINT (-1487762.628 1671209.907)
3,220253,400259501001199,1199,1,950100,25,40,POINT (-545769.871 1552960.53)
4,220254,483719501001162,1162,1,950100,371,48,POINT (-621720.17 923137.893)


# Load Features from US Census

2023 US Census Data

Using an API key, we will use the 'census' Python package to interact with the US Govermnent's census API.

In [6]:
census = load_census.CensusData(sql_obj=sql_obj, year=2023, granularity='county')

In [7]:
combined_gdf = census.merge_census_info(properties)
combined_gdf.head()


3090 geographies needed, 3082 cached, 8 to fetch.
Unique geoids: 8.
[1/8] Querying id 09011 at county...
[2/8] Querying id 09001 at county...
[3/8] Querying id 09003 at county...
[4/8] Querying id 09015 at county...
[5/8] Querying id 09005 at county...
  Time estimate: 1m 56s total (14.62s per geography, 43.9s remaining)
[6/8] Querying id 09009 at county...
[7/8] Querying id 09013 at county...
[8/8] Querying id 09007 at county...
Completed fetching census data in 2m 6s (8 API calls, 0 cache hits)
Dropped 306 properties in 8 geographies with no census data (e.g., Virginia independent cities).


,geoid,geometry,B01001A_001E,B01001A_002E,B01001A_003E,B01001A_004E,B01001A_005E,B01001A_006E,B01001A_007E,B01001A_008E,...,B25091_014E,B25091_015E,B25091_016E,B25091_017E,B25091_018E,B25091_019E,B25091_020E,B25091_021E,B25091_022E,B25091_023E
0,360259705012013,POINT (1679681.993 2303127.485),NaN,NaN,0.005931,0.006472,0.007577,0.004726,0.005572,0.008568,...,0.212319,0.107708,0.071783,0.032003,0.033655,0.015829,0.010736,0.013008,0.052099,0.009429
1,270753701012282,POINT (337884.435 2694375.859),NaN,NaN,0.007580,0.009158,0.009189,0.005724,0.001516,0.007735,...,0.213491,0.082130,0.039053,0.020118,0.023432,0.008757,0.004734,0.006864,0.039527,0.005444
2,040159501032038,POINT (-1487762.628 1671209.907),NaN,NaN,0.005709,0.006555,0.005608,0.003770,0.001846,0.005852,...,0.237095,0.091328,0.042595,0.025789,0.012292,0.014717,0.007597,0.009138,0.024864,0.013077
3,400259501001199,POINT (-545769.871 1552960.53),NaN,NaN,0.009146,0.007927,0.017683,0.000305,0.002591,0.000000,...,0.527675,0.121771,0.035055,0.003690,0.086716,0.000000,0.000000,0.016605,0.025830,0.016605
4,483719501001162,POINT (-621720.17 923137.893),NaN,NaN,0.007498,0.008952,0.009589,0.003817,0.001318,0.003954,...,0.333423,0.064638,0.033665,0.062752,0.003771,0.028010,0.000000,0.016698,0.048478,0.009426


# Load Wildfire GIS Data for 2024

We will use point data from the Visible Infrared Imaging Radiometer Suite (VIIRS). A valid alternative is using burn boundary data. There are a few different data sources we could use, but in the interest of (portfolio) simplicity we'll use just the VIIRS.

N:B: May be a good chance to practice using AWS DB storage and retrieval?

In [8]:
wildfires = load_wildfires.WildfireData(sql_obj=sql_obj)


Loading previously extracted wildfire data from: ['J1V-C2' 'J1V-C2,SV-C2' 'J1V-C2,J2V-C2' 'J1V-C2,J2V-C2,SV-C2' 'J2V-C2'
 'J2V-C2,SV-C2' 'SV-C2']


In [9]:
# Visualize wildfire locations
# wildfire_map = wildfires.visualize_data(save_path=Path("figures/wildfires_map.html"))
# wildfire_map

## Create Targets (Wildfire Proximity Score)

Give Each Property a Wildfire Risk Score based on the proximity to wildfires.

TODO: List/describe the various options given for targets.


In [10]:
proximity_features = gis.calc_all_features(combined_gdf, wildfires.data)
targets_features = pd.concat([combined_gdf, proximity_features], axis=1)

Loading cached GIS features from gis_features_afd39c32d782.parquet


In [11]:
targets_features.head()

,geoid,geometry,B01001A_001E,B01001A_002E,B01001A_003E,B01001A_004E,B01001A_005E,B01001A_006E,B01001A_007E,B01001A_008E,...,exp_decay_score,fire_count_0_10km,fire_count_10_25km,fire_count_25_50km,fire_count_50_100km,fire_FRP_0_10km,fire_FRP_10_25km,fire_FRP_25_50km,fire_FRP_50_100km,nearest_fire_km
0,360259705012013,POINT (1679681.993 2303127.485),NaN,NaN,0.005931,0.006472,0.007577,0.004726,0.005572,0.008568,...,53.983695,0.0,0.0,21.0,21.0,0.0,0.0,274.05,161.490000,48.349709
1,270753701012282,POINT (337884.435 2694375.859),NaN,NaN,0.007580,0.009158,0.009189,0.005724,0.001516,0.007735,...,450.516230,0.0,0.0,0.0,252.0,0.0,0.0,0.00,10755.255000,57.679484
2,040159501032038,POINT (-1487762.628 1671209.907),NaN,NaN,0.005709,0.006555,0.005608,0.003770,0.001846,0.005852,...,1816.634864,0.0,0.0,0.0,483.0,0.0,0.0,0.00,32008.411772,56.598821
3,400259501001199,POINT (-545769.871 1552960.53),NaN,NaN,0.009146,0.007927,0.017683,0.000305,0.002591,0.000000,...,2212.463405,0.0,0.0,105.0,315.0,0.0,0.0,3871.49,19067.378750,25.247955
4,483719501001162,POINT (-621720.17 923137.893),NaN,NaN,0.007498,0.008952,0.009589,0.003817,0.001318,0.003954,...,1250.572963,0.0,0.0,63.0,1680.0,0.0,0.0,420.84,30745.190000,29.513294


In [12]:
# # Visualize properties colored by wildfire risk, with wildfire locations
# combined_map = visualize.create_combined_map(
#     targets_features,
#     wildfires.data,
#     risk_column="nearest_fire_km",
#     save_path=Path("figures/risk_map.html")
# )
# combined_map

# Machine Learning Considerations
## Scoring Methods

For the float risk score, we can use Mean Squared Error (MSE) or Root Mean Squared Error (RMSE). Since it's quadratic in difference between observations and predictions deviations, MSE strongly penalizes large misses, which would be expensive for the insurance company.

For the risk category counts, they appear to be Poisson distributed, so a Poisson loss-function is appropriate.

For any classification model with the binned risk categories, we want to make large misses costly (i.e. predicting a 1 when the category is a 10), since these would also be very costly to the insurance company. To be honest, MSE will work here as well, since the categories are just 

# Model Machine Learning


NB: A good chance to make use of AWS compute.


### Split Data into Features/Targets

We use `nearest_fire_km` as the target — the distance in kilometres to the nearest wildfire detection. With the full 300k-property dataset, `exp_decay_score` (which captures both proximity and density of nearby fires) would be a better choice, but the small 119-property test set has nearly zero variance in decay score because all properties are ~360 km from the nearest fire.

All other proximity-derived columns are dropped so the model only sees census features as inputs.

In [13]:
TARGET_COL = "nearest_fire_km"

# All proximity features are derived from the same wildfire data — drop them
# so the model only sees census features as inputs.
proximity_cols = [c for c in proximity_features.columns]
drop_cols = ["geometry", "geoid"] + [c for c in proximity_cols if c != TARGET_COL]

# SAVE TO PARQUET FOR AWS
targets_features.drop(columns=drop_cols).to_parquet(PATH_DATA/"model_joined.parquet")

# Preprocessing with adaptive imputation based on missingness analysis
# - MCAR features: SimpleImputer (median) - fast
# - MAR features: IterativeImputer - preserves correlations
X_train, X_test, y_train, y_test, feature_names, pipeline = train.preprocess_with_cache(
    targets_features,
    TARGET_COL,
    drop_cols,
    nan_threshold=0.45,
    corr_threshold=0.85,
    mar_corr_threshold=0.1,  # Features with missingness corr > 0.1 use IterativeImputer
    use_cache=True,
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target range: {y_train.min():.4f} – {y_train.max():.4f}")
print(f"Target std:   {y_train.std():.6f}")
print(f"Target mean:  {y_train.mean():.4f}")

Loading cached preprocessing from preprocess_ae4cc0cd8312.pkl
  Loaded 633 features from cache
Train: (132891, 633), Test: (33223, 633)
Target range: 0.0289 – 315.5402
Target std:   31.987690
Target mean:  28.6079


In [14]:
# Diagnostic: check if the target has meaningful variance
cv = y_train.std() / y_train.mean() * 100  # coefficient of variation
print(f"Coefficient of variation: {cv:.4f}%")
if cv < 1.0:
    print(
        f"WARNING: Target has near-zero variance (CV={cv:.4f}%). "
        f"All properties are ~{y_train.mean():.1f} km from the nearest fire. "
        f"Models will appear to have perfect accuracy but are not learning meaningful patterns. "
        f"Scale to 300k properties for geographic diversity."
    )

Coefficient of variation: 111.8143%


### Preprocessing

Drop high-NaN columns, remove correlated features, then impute + scale. All steps are fit on training data only to prevent leakage.

**Adaptive Imputation:** The preprocessing analyzes each feature's missingness mechanism:
- **MCAR** (Missing Completely At Random): Uses `SimpleImputer(median)` - fast, O(n)
- **MAR** (Missing At Random): Uses `IterativeImputer` - preserves feature correlations

**Caching:** Results are cached to `data/cache/preprocess_{hash}.pkl`. Re-runs with unchanged data load from cache in <1s.

In [15]:
# Preprocessing is now handled by train.preprocess_with_cache() above
# which performs:
#   1. Missingness analysis (MCAR vs MAR detection)
#   2. Adaptive imputation (SimpleImputer for MCAR, IterativeImputer for MAR)
#   3. StandardScaler
# All results are cached - re-runs with unchanged data load in <1s

## Train Models


#### RandomForestRegressor


In [16]:
# rfr_search = train.train_random_forest(X_train, y_train, n_iter=20, cv=5)
# print(f"Best RF params: {rfr_search.best_params_}")
#
# rfr_metrics = train.evaluate_model(rfr_search.best_estimator_, X_train, X_test, y_train, y_test)
# print(f"RF Train RMSE: {rfr_metrics['train_rmse']:.8f}")
# print(f"RF Test  RMSE: {rfr_metrics['test_rmse']:.8f}")

#### XGBoost


In [ ]:
xgb_search = train.train_xgboost(X_train, y_train, n_iter=20, cv=5)
print(f"Best XGB params: {xgb_search.best_params_}")

xgb_metrics = train.evaluate_model(xgb_search.best_estimator_, X_train, X_test, y_train, y_test)
print(f"XGB Train RMSE: {xgb_metrics['train_rmse']:.8f}")
print(f"XGB Test  RMSE: {xgb_metrics['test_rmse']:.8f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


#### Extract Feature Weights





In [ ]:
# Pick the better model
# if xgb_metrics["test_rmse"] <= rfr_metrics["test_rmse"]:
best_model = xgb_search.best_estimator_
print("Best model: XGBoost")
# else:
#     best_model = rfr_search.best_estimator_
#     print("Best model: RandomForest")

top_features = train.extract_feature_importance(best_model, feature_names, top_n=10)
print(f"\nTop 10 features:\n{top_features}")

In [ ]:
# Feature importance bar chart
fig_importance = visualize.plot_feature_importance(
    top_features, 
    title="Top 10 Feature Importances",
    save_path=Path("figures/feature_importance.png")
)
fig_importance

In [ ]:
# Actual vs Predicted scatter plot
y_pred = best_model.predict(X_test)

fig_scatter = visualize.plot_actual_vs_predicted(
    y_test.values, 
    y_pred,
    title="Actual vs Predicted (Test Set)",
    xlabel="Actual Distance to Fire (km)",
    ylabel="Predicted Distance to Fire (km)",
    save_path=Path("figures/actual_vs_predicted.png")
)
fig_scatter

In [ ]:
model_path = Path("Models") / "best_model.pkl"
train.save_model(best_model, model_path, pipeline=pipeline, feature_names=feature_names)
print(f"Model saved to {model_path}")

# Conclusion